In [54]:
# Install & Import

import json
import re
import random
from collections import Counter

In [55]:
# Import NCBI MACCROBAT2018 Dataset
import os

def load_macrobot2018(folder_path):
    """Load .ann and .txt files from MACCROBAT2018 dataset"""
    files = sorted([f.replace('.ann', '') for f in os.listdir(folder_path) if f.endswith('.ann')])
    
    data = []
    for file_id in files:
        txt_path = os.path.join(folder_path, f"{file_id}.txt")
        ann_path = os.path.join(folder_path, f"{file_id}.ann")
        
        # Read text
        with open(txt_path, 'r', encoding='utf-8') as f:
            text = f.read()
        
        # Read annotations
        entities = []
        with open(ann_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.startswith('T'):  # Entity annotation
                    parts = line.strip().split('\t')
                    if len(parts) >= 2:
                        info = parts[1].split()
                        label = info[0]
                        # Handle multiple spans (e.g., "1373;1386") - take first span
                        start = int(info[1].split(';')[0])
                        end = int(info[2].split(';')[0])
                        entities.append({
                            "start": start,
                            "end": end,
                            "label": label
                        })
        
        data.append({"text": text, "entities": entities})
    
    return data

# Load the dataset
data_path = r"C:\Users\sanan\OneDrive\Documents\NLP\NLP Capstone Project\MACCROBAT2018"
dataset = load_macrobot2018(data_path)
print(f"Loaded {len(dataset)} documents")
print(f"Sample: {dataset[0]}")

Loaded 200 documents
Sample: {'text': "CASE: A 28-year-old previously healthy man presented with a 6-week history of palpitations.\nThe symptoms occurred during rest, 2–3 times per week, lasted up to 30 minutes at a time and were associated with dyspnea.\nExcept for a grade 2/6 holosystolic tricuspid regurgitation murmur (best heard at the left sternal border with inspiratory accentuation), physical examination yielded unremarkable findings.\nAn electrocardiogram (ECG) revealed normal sinus rhythm and a Wolff– Parkinson– White pre-excitation pattern (Fig.1: Top), produced by a right-sided accessory pathway.\nTransthoracic echocardiography demonstrated the presence of Ebstein's anomaly of the tricuspid valve, with apical displacement of the valve and formation of an “atrialized” right ventricle (a functional unit between the right atrium and the inlet [inflow] portion of the right ventricle) (Fig.2).\nThe anterior tricuspid valve leaflet was elongated (Fig.2C, arrow), whereas the septal

In [56]:
# Explore the loaded dataset
print(f"Total documents: {len(dataset)}")

# Show entity types
entity_types = []
for doc in dataset:
    for ent in doc["entities"]:
        entity_types.append(ent["label"])

from collections import Counter
entity_counts = Counter(entity_types)
print(f"\nEntity type distribution:")
for label, count in entity_counts.most_common():
    print(f"  {label}: {count}")

# Show sample document
print(f"\nSample document:")
print(f"Text length: {len(dataset[0]['text'])}")
print(f"Entities: {len(dataset[0]['entities'])}")
print(f"Text preview: {dataset[0]['text'][:200]}...")

Total documents: 200

Entity type distribution:
  Diagnostic_procedure: 4567
  Sign_symptom: 3359
  Biological_structure: 2931
  Detailed_description: 2901
  Lab_value: 2858
  Disease_disorder: 1362
  Medication: 1076
  Therapeutic_procedure: 1005
  Date: 731
  Clinical_event: 626
  History: 392
  Severity: 369
  Dosage: 362
  Nonbiological_location: 354
  Coreference: 313
  Duration: 280
  Age: 206
  Sex: 191
  Administration: 175
  Distance: 122
  Activity: 108
  Family_history: 81
  Frequency: 76
  Shape: 65
  Personal_background: 57
  Time: 57
  Subject: 54
  Color: 52
  Texture: 46
  Area: 43
  Outcome: 42
  Qualitative_concept: 41
  Volume: 33
  Quantitative_concept: 31
  Other_event: 22
  Other_entity: 20
  Occupation: 13
  Biological_attribute: 10
  Height: 4
  Weight: 4
  Mass: 2

Sample document:
Text length: 1686
Entities: 68
Text preview: CASE: A 28-year-old previously healthy man presented with a 6-week history of palpitations.
The symptoms occurred during rest, 2–3 times 

Tokenization + BIO Conversion

In [57]:
def tokenize_with_offsets(text):
    return [(m.group(), m.start(), m.end())
            for m in re.finditer(r'\w+|\S', text)]


def convert_to_bio(text, entities):
    tokens_with_offsets = tokenize_with_offsets(text)
    tags = ["O"] * len(tokens_with_offsets)

    for ent in entities:
        ent_start, ent_end, label = ent["start"], ent["end"], ent["label"]

        entity_token_indices = []
        for i, (token_text, token_start, token_end) in enumerate(tokens_with_offsets):
            # Check for overlap: token_start < ent_end AND token_end > ent_start
            # This condition means the token's span intersects with the entity's span.
            if token_start < ent_end and token_end > ent_start:
                entity_token_indices.append(i)

        if not entity_token_indices:
            continue

        # Mark the first token of the entity as B-label
        tags[entity_token_indices[0]] = f"B-{label}"

        # Mark subsequent tokens of the entity as I-label
        for i in range(1, len(entity_token_indices)):
            tags[entity_token_indices[i]] = f"I-{label}"

    return [(tokens_with_offsets[i][0], tags[i]) for i in range(len(tokens_with_offsets))]


Load & Convert Dataset


In [58]:
def load_and_convert(json_file):
    with open(json_file, "r") as f:
        data = json.load(f)

    sentences = []
    for item in data:
        bio = convert_to_bio(item["text"], item["entities"])
        sentences.append(bio)

    return sentences


def save_bio(sentences, filename):
    with open(filename, "w") as f:
        for sent in sentences:
            for token, tag in sent:
                f.write(f"{token} {tag}\n")
            f.write("\n")

Split Dataset

In [59]:
def split_data(sentences, train=0.7, val=0.1, test=0.2, seed=42):
    random.seed(seed)
    random.shuffle(sentences)

    n = len(sentences)
    t = int(n * train)
    v = int(n * val)

    return (
        sentences[:t],
        sentences[t:t+v],
        sentences[t+v:]
    )

Load BIO into Tokens

In [60]:
def load_bio(filepath):
    sentences, tags = [], []
    s, t = [], []

    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line:
                if s:
                    sentences.append(s)
                    tags.append(t)
                    s, t = [], []
            else:
                tok, tag = line.split()
                s.append(tok)
                t.append(tag)

    if s:
        sentences.append(s)
        tags.append(t)

    return sentences, tags

Preprocessing

In [61]:
def clean_token(token):
    token = token.lower()
    token = re.sub(r'\d', '0', token)
    return token


def preprocess(sentences):
    return [[clean_token(w) for w in sent] for sent in sentences]

Build Vocabulary

In [62]:
def build_vocab(sentences, tags):
    word_counter = Counter(w for s in sentences for w in s)
    tag_set = set(t for seq in tags for t in seq)

    word2idx = {"<PAD>": 0, "<UNK>": 1}
    for w in word_counter:
        word2idx[w] = len(word2idx)

    tag2idx = {"<PAD>": 0}
    for t in sorted(tag_set):
        tag2idx[t] = len(tag2idx)

    char2idx = {"<PAD>": 0, "<UNK>": 1}
    for w in word_counter:
        for ch in w:
            if ch not in char2idx:
                char2idx[ch] = len(char2idx)

    return word2idx, tag2idx, char2idx

Encoding

In [63]:
def encode(sentences, tags, word2idx, tag2idx, char2idx):
    X, y, X_char = [], [], []

    for sent, tag_seq in zip(sentences, tags):
        word_ids = [word2idx.get(w, 1) for w in sent]
        tag_ids = [tag2idx[t] for t in tag_seq]

        char_ids = []
        for w in sent:
            char_ids.append([char2idx.get(c, 1) for c in w])

        X.append(word_ids)
        y.append(tag_ids)
        X_char.append(char_ids)

    return X, y, X_char

Padding

In [64]:
def pad(seq, max_len, pad_val=0):
    return [s[:max_len] + [pad_val]*(max_len-len(s)) for s in seq]


def pad_chars(seq, max_len, max_word_len):
    out = []
    for sent in seq:
        s = []
        for w in sent:
            w = w[:max_word_len] + [0]*(max_word_len-len(w))
            s.append(w)
        while len(s) < max_len:
            s.append([0]*max_word_len)
        out.append(s[:max_len])
    return out

Mask

In [65]:
def create_mask(X):
    return [[1 if tok != 0 else 0 for tok in seq] for seq in X]

Run Full Pipeline

In [66]:
import numpy as np
from collections import defaultdict

# Tokenization with exact offsets, to align entity spans correctly.
def tokenize_with_offsets(text):
    return [(m.group(), m.start(), m.end())
            for m in re.finditer(r"\w+|[^\w\s]", text)]


def convert_to_bio(text, entities):
    tokens = tokenize_with_offsets(text)
    tags = ["O"] * len(tokens)

    for ent in sorted(entities, key=lambda e: e["start"]):
        ent_start, ent_end, label = ent["start"], ent["end"], ent["label"]
        entity_token_indices = [i for i, (_, start, end) in enumerate(tokens)
                                if start < ent_end and end > ent_start]

        if not entity_token_indices:
            continue

        tags[entity_token_indices[0]] = f"B-{label}"
        for idx in entity_token_indices[1:]:
            tags[idx] = f"I-{label}"

    return [(tokens[i][0], tags[i]) for i in range(len(tokens))]


def save_bio(sentences, filepath):
    with open(filepath, "w", encoding="utf-8") as f:
        for sent in sentences:
            for token, tag in sent:
                f.write(f"{token} {tag}\n")
            f.write("\n")


def split_data(data, train_ratio=0.8, val_ratio=0.1, seed=42):
    random.Random(seed).shuffle(data)
    n = len(data)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))
    return data[:train_end], data[train_end:val_end], data[val_end:]


def load_bio(filepath):
    sentences, tags = [], []
    cur_sent, cur_tags = [], []

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if line == "":
                if cur_sent:
                    sentences.append(cur_sent)
                    tags.append(cur_tags)
                    cur_sent, cur_tags = [], []
            else:
                parts = line.split()
                if len(parts) == 2:
                    cur_sent.append(parts[0])
                    cur_tags.append(parts[1])

    if cur_sent:
        sentences.append(cur_sent)
        tags.append(cur_tags)

    return sentences, tags


def preprocess(sentences):
    processed = []
    for sent in sentences:
        new_sent = []
        for token in sent:
            token = token.lower().strip()
            token = re.sub(r"\d", "0", token)
            new_sent.append(token if token else "<unk>")
        processed.append(new_sent)
    return processed


def build_vocab(sentences, tags):
    word_freq = defaultdict(int)
    tag_set = set()
    char_set = set()
    for sent, sent_tags in zip(sentences, tags):
        for token, tag in zip(sent, sent_tags):
            word_freq[token] += 1
            tag_set.add(tag)
            for ch in token:
                char_set.add(ch)

    word2idx = {"<pad>": 0, "<unk>": 1}
    for word in sorted(word_freq):
        word2idx[word] = len(word2idx)

    tag2idx = {"<pad>": 0}
    for tag in sorted(tag_set):
        tag2idx[tag] = len(tag2idx)

    char2idx = {"<pad>": 0, "<unk>": 1}
    for ch in sorted(char_set):
        char2idx[ch] = len(char2idx)

    return word2idx, tag2idx, char2idx


def encode(sentences, tags, word2idx, tag2idx, char2idx):
    X, y, X_char = [], [], []
    for sent, sent_tags in zip(sentences, tags):
        word_ids = [word2idx.get(t, word2idx["<unk>"]) for t in sent]
        tag_ids = [tag2idx.get(t, 0) for t in sent_tags]
        char_ids = [[char2idx.get(ch, char2idx["<unk>"]) for ch in token] for token in sent]
        X.append(word_ids)
        y.append(tag_ids)
        X_char.append(char_ids)
    return X, y, X_char


def pad(sequences, max_len, pad_value=0):
    padded = []
    for seq in sequences:
        seq = seq[:max_len]
        seq = seq + [pad_value] * (max_len - len(seq))
        padded.append(seq)
    return np.array(padded, dtype=np.int32)


def pad_chars(sequences, max_len, max_word_len, pad_value=0):
    padded = []
    for sent in sequences:
        sent = sent[:max_len]
        padded_sent = []
        for word in sent:
            word = word[:max_word_len]
            word = word + [pad_value] * (max_word_len - len(word))
            padded_sent.append(word)
        while len(padded_sent) < max_len:
            padded_sent.append([pad_value] * max_word_len)
        padded.append(padded_sent)
    return np.array(padded, dtype=np.int32)


def create_mask(X):
    return (X != 0).astype(np.int32)

# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# PIPELINE
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

# Step 1: Convert dataset to BIO format
bio_data = []
for item in dataset:
    bio = convert_to_bio(item["text"], item["entities"])
    bio_data.append(bio)

save_bio(bio_data, "bio.txt")
print(f"âœ… Converted {len(bio_data)} documents to BIO format")

# Step 2: Split into train / val / test
train, val, test = split_data(bio_data, train_ratio=0.8, val_ratio=0.1, seed=42)
save_bio(train, "train.txt")
save_bio(val,   "val.txt")
save_bio(test,  "test.txt")
print(f"âœ… Split data: Train={len(train)}, Val={len(val)}, Test={len(test)}")

# Step 3: Load train data
train_sents, train_tags = load_bio("train.txt")
print(f"âœ… Loaded train: {len(train_sents)} sentences")

# Step 4: Preprocess
train_sents = preprocess(train_sents)
print(f"âœ… Preprocessed tokens")

# Step 5: Build vocabulary
word2idx, tag2idx, char2idx = build_vocab(train_sents, train_tags)
print(f"âœ… Built vocabulary: Words={len(word2idx)}, Tags={len(tag2idx)}, Chars={len(char2idx)}")

# Step 6: Encode
X, y, X_char = encode(train_sents, train_tags, word2idx, tag2idx, char2idx)
print(f"âœ… Encoded data")

# Step 7: Padding
MAX_LEN      = max(len(s) for s in X)
MAX_WORD_LEN = max(len(w) for s in X_char for w in s)

X      = pad(X, MAX_LEN)
y      = pad(y, MAX_LEN)
X_char = pad_chars(X_char, MAX_LEN, MAX_WORD_LEN)
print(f"âœ… Padded: Max Length={MAX_LEN}, Max Word Length={MAX_WORD_LEN}")

# Step 8: Create mask
mask = create_mask(X)
print(f"âœ… Created attention mask")

print("\n" + "=" * 50)
print("âœ… PIPELINE COMPLETED")
print("=" * 50)
print(f"Total sentences     : {len(X)}")
print(f"Max sequence length : {MAX_LEN}")
print(f"Word vocabulary size: {len(word2idx)}")
print(f"Tag vocabulary size : {len(tag2idx)}")
print(f"Char vocabulary size: {len(char2idx)}")

âœ… Converted 200 documents to BIO format
âœ… Split data: Train=160, Val=20, Test=20
âœ… Loaded train: 160 sentences
âœ… Preprocessed tokens
âœ… Built vocabulary: Words=7600, Tags=83, Chars=82
âœ… Encoded data
âœ… Padded: Max Length=1213, Max Word Length=26
âœ… Created attention mask

âœ… PIPELINE COMPLETED
Total sentences     : 160
Max sequence length : 1213
Word vocabulary size: 7600
Tag vocabulary size : 83
Char vocabulary size: 82


In [67]:
# Load and process validation data
val_sents, val_tags = load_bio("val.txt")
val_sents = preprocess(val_sents)
X_val, y_val, X_char_val = encode(val_sents, val_tags, word2idx, tag2idx, char2idx)
X_val = pad(X_val, MAX_LEN)
y_val = pad(y_val, MAX_LEN)
X_char_val = pad_chars(X_char_val, MAX_LEN, MAX_WORD_LEN)
mask_val = create_mask(X_val)
print(f"âœ… Loaded and processed validation data: {len(X_val)} sentences")

âœ… Loaded and processed validation data: 20 sentences


In [68]:
# Shape Consistency Check
len(X) == len(y)

True

In [69]:
# Tag Integrity Check

idx2tag = {v: k for k, v in tag2idx.items()}

for seq in y:
    for i, tag_id in enumerate(seq):
        tag = idx2tag[tag_id]
        # Handle '<PAD>' tags which should not be checked for BIO integrity
        if tag == '<PAD>':
            continue

        if tag.startswith("I"):
            # If it's an 'I' tag at the beginning of a sequence or preceded by 'O'
            # this is an invalid BIO sequence.
            if i == 0 or idx2tag[seq[i-1]] == "O":
                print(f"Invalid BIO sequence found: 'I' tag '{tag}' at index {i} following '{idx2tag[seq[i-1]]}' (or at start of sequence).")


In [70]:
# Vocabulary Coverage

unknown_ratio = sum(1 for sent in X for w in sent if w == 1) / sum(len(s) for s in X)
print("UNK ratio:", unknown_ratio)

UNK ratio: 0.0


In [71]:
# Padding & Mask Check

print(mask[0])

[1 1 1 ... 0 0 0]


In [72]:
# Label Distribution

from collections import Counter
print(Counter(tag for seq in train_tags for tag in seq))

Counter({'O': 45340, 'I-Lab_value': 3935, 'B-Diagnostic_procedure': 3718, 'I-Diagnostic_procedure': 3510, 'B-Sign_symptom': 2661, 'I-Detailed_description': 2429, 'B-Biological_structure': 2360, 'B-Detailed_description': 2337, 'B-Lab_value': 2299, 'I-Biological_structure': 2077, 'I-Sign_symptom': 1233, 'I-Date': 1230, 'B-Disease_disorder': 1097, 'I-History': 1033, 'I-Dosage': 917, 'B-Medication': 908, 'B-Therapeutic_procedure': 795, 'I-Disease_disorder': 724, 'B-Date': 622, 'I-Age': 617, 'I-Therapeutic_procedure': 499, 'B-Clinical_event': 493, 'I-Duration': 409, 'I-Medication': 402, 'B-Dosage': 321, 'I-Other_entity': 315, 'I-Family_history': 308, 'B-Severity': 298, 'B-History': 297, 'B-Nonbiological_location': 282, 'I-Nonbiological_location': 280, 'I-Distance': 258, 'B-Coreference': 228, 'B-Duration': 227, 'I-Area': 187, 'I-Clinical_event': 171, 'B-Age': 164, 'B-Sex': 153, 'B-Administration': 151, 'I-Volume': 145, 'I-Coreference': 103, 'B-Distance': 103, 'I-Time': 102, 'I-Frequency': 95

Word Embedding Layer (PyTorch)

In [73]:
import torch
import torch.nn as nn

class WordEmbedding(nn.Module):
    def __init__(self, vocab_size, embed_dim, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_idx
        )

    def forward(self, x):
        return self.embedding(x)

In [74]:
#Usage

vocab_size = len(word2idx)
embed_dim = 100

word_embed = WordEmbedding(vocab_size, embed_dim)

sample = torch.tensor(X[:2])  # batch of sentences
out = word_embed(sample)

print(out.shape)
# (batch_size, seq_len, embed_dim)

torch.Size([2, 1213, 100])


In [75]:
glove_path = r"E:\DUK\S2\NLP\project\glove"

Load Pretrained GloVe Embeddings

In [76]:
# Load GloVe File

import numpy as np
import torch

def load_glove_embeddings(glove_path):
    glove_dict = {}
    
    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.strip().split()
            word = values[0]
            vector = np.array(values[1:], dtype='float32')
            glove_dict[word] = vector

    print(f"Loaded {len(glove_dict)} GloVe vectors")
    return glove_dict

In [77]:
# Create Embedding Matrix

def create_embedding_matrix(word2idx, glove_dict, embed_dim=100):
    vocab_size = len(word2idx)
    
    embedding_matrix = np.random.uniform(-0.25, 0.25, (vocab_size, embed_dim))

    found = 0
    for word, idx in word2idx.items():
        if word in glove_dict:
            embedding_matrix[idx] = glove_dict[word]
            found += 1

    print(f"Matched {found}/{vocab_size} words with GloVe")
    
    return torch.tensor(embedding_matrix, dtype=torch.float32)

In [78]:
# Create Word Embedding Layer

import torch.nn as nn

def build_word_embedding_layer(embedding_matrix):
    return nn.Embedding.from_pretrained(
        embedding_matrix,
        freeze=False,      # allow training
        padding_idx=0
    )

In [79]:
# Usage

glove_path = r"C:\Users\sanan\OneDrive\Documents\NLP\NLP Capstone Project\GloVe\glove.6B.100d.txt"

glove_dict = load_glove_embeddings(glove_path)
embedding_matrix = create_embedding_matrix(word2idx, glove_dict, embed_dim=100)

word_embedding = build_word_embedding_layer(embedding_matrix)

Loaded 400000 GloVe vectors
Matched 6321/7600 words with GloVe


Character Embedding Layer (BiLSTM)

In [80]:
class CharEmbedding(nn.Module):
    def __init__(self, char_vocab_size, char_embed_dim=30, hidden_dim=50):
        super(CharEmbedding, self).__init__()
        
        self.char_embed = nn.Embedding(char_vocab_size, char_embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size=char_embed_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

    def forward(self, x):
        # x: (batch_size, seq_len, word_len)
        batch_size, seq_len, word_len = x.size()

        x = x.view(-1, word_len)  # (batch*seq_len, word_len)
        x = self.char_embed(x)    # (batch*seq_len, word_len, char_embed_dim)

        _, (h, _) = self.lstm(x)

        # Concatenate forward + backward
        h = torch.cat((h[0], h[1]), dim=1)

        return h.view(batch_size, seq_len, -1)

In [81]:
# Usage

char_model = CharEmbedding(len(char2idx))

sample_char = torch.tensor(X_char[:2])  # batch
char_out = char_model(sample_char)

print(char_out.shape)
# (batch_size, seq_len, 100)  â†’ because bidirectional (50*2)

torch.Size([2, 1213, 100])


Combine Word + Character Embeddings

In [82]:
class CombinedEmbedding(nn.Module):
    def __init__(self, word_embedding_layer, char_vocab_size):
        super(CombinedEmbedding, self).__init__()

        self.word_embed = word_embedding_layer
        self.char_embed = CharEmbedding(char_vocab_size)

    def forward(self, word_input, char_input):
        word_vec = self.word_embed(word_input)     # (B, L, 100)
        char_vec = self.char_embed(char_input)     # (B, L, 100)

        combined = torch.cat([word_vec, char_vec], dim=-1)

        return combined

In [83]:
# Usage

model_embed = CombinedEmbedding(word_embedding, len(char2idx))

word_tensor = torch.tensor(X[:2])
char_tensor = torch.tensor(X_char[:2])

output = model_embed(word_tensor, char_tensor)

print(output.shape)
# (batch_size, seq_len, 200)

torch.Size([2, 1213, 200])


Sanity Check

In [84]:
print("Word Embedding Shape:", word_embedding(word_tensor).shape)
print("Char Embedding Shape:", char_model(char_tensor).shape)
print("Combined Shape:", output.shape)

Word Embedding Shape: torch.Size([2, 1213, 100])
Char Embedding Shape: torch.Size([2, 1213, 100])
Combined Shape: torch.Size([2, 1213, 200])


In [85]:
%pip install pytorch-crf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


BUILD THE MODEL

In [86]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchcrf import CRF

# -------------------------------
# Device
# -------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------------
# Dataset
# -------------------------------
class NERDataset(Dataset):
    def __init__(self, X, X_char, y, mask):
        self.X = torch.tensor(X, dtype=torch.long)
        self.X_char = torch.tensor(X_char, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)
        self.mask = torch.tensor(mask, dtype=torch.bool)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.X_char[idx], self.y[idx], self.mask[idx]

# -------------------------------
# Model
# -------------------------------
class BiLSTM_CRF(nn.Module):
    def __init__(self, word_embedding, char_vocab_size, tagset_size):
        super().__init__()

        self.word_embed = word_embedding

        self.char_embed = nn.Embedding(char_vocab_size, 30, padding_idx=0)
        self.char_lstm = nn.LSTM(30, 50, batch_first=True, bidirectional=True)

        self.bilstm = nn.LSTM(
            input_size=100 + 100,   # word(100) + char(50*2)
            hidden_size=100,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(200, tagset_size)
        self.crf = CRF(tagset_size, batch_first=True)

    def forward(self, X, X_char, tags=None, mask=None):
        # Word embeddings
        word_vec = self.word_embed(X)

        # Char embeddings
        B, L, W = X_char.size()
        X_char = X_char.view(-1, W)

        char_vec = self.char_embed(X_char)
        _, (h, _) = self.char_lstm(char_vec)

        char_vec = torch.cat((h[0], h[1]), dim=1)
        char_vec = char_vec.view(B, L, -1)

        # Combine
        x = torch.cat([word_vec, char_vec], dim=-1)

        # BiLSTM
        x, _ = self.bilstm(x)
        x = self.dropout(x)

        emissions = self.fc(x)

        # CRF
        if tags is not None:
            return -self.crf(emissions, tags, mask=mask, reduction='mean')
        else:
            return self.crf.decode(emissions, mask=mask)

# -------------------------------
# Data Loaders
# -------------------------------
batch_size = 16

train_loader = DataLoader(
    NERDataset(X, X_char, y, mask),
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    NERDataset(X_val, X_char_val, y_val, mask_val),
    batch_size=batch_size
)

# -------------------------------
# Model Init
# -------------------------------
model = BiLSTM_CRF(
    word_embedding,
    char_vocab_size=len(char2idx),
    tagset_size=len(tag2idx)
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


TRAIN THE MODEL

In [ ]:
%pip install transformers datasets huggingface-hub

import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForTokenClassification, Trainer, TrainingArguments
from datasets import Dataset
import numpy as np

# =============================
# TRANSFORMER-BASED NER MODEL (BioBERT)
# =============================
print("⚙️ Setting up Transformer-based NER using BioBERT...")

# Model: BioBERT (pretrained on biomedical literature)
model_name = "dmis-lab/biobert-base-cased-v1.2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Create label2id and id2label mappings
label2id = {label: idx for idx, label in enumerate(sorted(tag2idx.keys()))}
id2label = {idx: label for label, idx in label2id.items()}

print(f"✅ Loaded BioBERT tokenizer from {model_name}")
print(f"📊 Total labels: {len(label2id)}")

# Prepare dataset for Hugging Face format
def prepare_biobert_dataset(sentences, tags, tokenizer, label2id, max_length=512):
    """Prepare sentences and tags for BioBERT fine-tuning."""
    encodings = tokenizer(
        sentences,
        is_split_into_words=True,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    
    labels = []
    for i in range(len(sentences)):
        word_ids = encodings.word_ids(i)
        label_ids = []
        previous_word_id = None
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != previous_word_id:
                label_ids.append(label2id[tags[i][word_id]])
                previous_word_id = word_id
            else:
                label_ids.append(-100)
        labels.append(label_ids)
    
    encodings["labels"] = labels
    return encodings

# Load and preprocess training data
train_sents_raw, train_tags_raw = load_bio("train.txt")
train_sents_raw = preprocess(train_sents_raw)

# Tokenize and prepare
print("📝 Preparing training data for BioBERT...")
train_encodings = prepare_biobert_dataset(train_sents_raw, train_tags_raw, tokenizer, label2id)

# Convert to HuggingFace Dataset
train_dataset = Dataset.from_dict(train_encodings)

print(f"✅ Prepared training dataset with {len(train_dataset)} samples")


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
    --------------------------------------- 0.3/10.6 MB ? eta -:--:--
   -- ------------------------------------- 0.8/10.6 MB 2.4 MB/s eta 0:00:05
   ---- ----------------------------------- 1.3/10.6 MB 2.2 MB/s eta 0:00:05
   ------ --------------------------------- 1.8/10.6 MB 2.2 MB/s eta 0:00:04
   -------- ------------------------------- 2.4/10.6 MB 2.3 MB/s eta 0:00:04
 

config.json: 0.00B [00:00, ?B/s]

c:\Users\sanan\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sanan\.cache\huggingface\hub\models--dmis-lab--biobert-base-cased-v1.2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt: 0.00B [00:00, ?B/s]

✅ Loaded BioBERT tokenizer from dmis-lab/biobert-base-cased-v1.2
📊 Total labels: 83
📝 Preparing training data for BioBERT...
✅ Prepared training dataset with 160 samples


In [98]:
# =============================
# TRAIN BIOBERT MODEL
# =============================
print("🚀 Loading BioBERT model for fine-tuning...")

biobert_model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    label2id=label2id,
    id2label=id2label
)

# Training arguments
training_args = TrainingArguments(
    output_dir="./biobert_ner_model",
    eval_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    push_to_hub=False,
)

# Prepare validation dataset
val_sents_raw, val_tags_raw = load_bio("val.txt")
val_sents_raw = preprocess(val_sents_raw)
val_encodings = prepare_biobert_dataset(val_sents_raw, val_tags_raw, tokenizer, label2id)
val_dataset = Dataset.from_dict(val_encodings)

print(f"📊 Validation dataset: {len(val_dataset)} samples")

# Define metrics function
def compute_metrics(p):
    from seqeval.metrics import f1_score, precision_score, recall_score
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    
    # Remove ignored indices
    true_predictions = [[id2label[p] for (p, l) in zip(prediction, label) if l != -100]
                        for prediction, label in zip(predictions, labels)]
    true_labels = [[id2label[l] for (p, l) in zip(prediction, label) if l != -100]
                   for prediction, label in zip(predictions, labels)]
    
    precision = precision_score(true_labels, true_predictions)
    recall = recall_score(true_labels, true_predictions)
    f1 = f1_score(true_labels, true_predictions)
    
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

# Initialize trainer
trainer = Trainer(
    model=biobert_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("⏳ Training BioBERT model...")
trainer.train()

# Save best model
biobert_model.save_pretrained("./biobert_ner_best")
tokenizer.save_pretrained("./biobert_ner_best")

print("✅ BioBERT model training completed and saved!")


🚀 Loading BioBERT model for fine-tuning...


[transformers] You passed `num_labels=83` which is incompatible to the `id2label` map of length `2`.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	

ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=1.1.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=1.1.0'`

In [99]:
# =============================
# EVALUATE BIOBERT vs BiLSTM+CRF
# =============================
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

def evaluate_biobert(model, tokenizer, test_sents, test_tags, id2label, batch_size=8):
    """Evaluate BioBERT model on test set."""
    model.eval()
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for i in range(0, len(test_sents), batch_size):
            batch_sents = test_sents[i:i+batch_size]
            batch_tags = test_tags[i:i+batch_size]
            
            encodings = tokenizer(
                batch_sents,
                is_split_into_words=True,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt"
            )
            
            input_ids = encodings["input_ids"].to(device)
            attention_mask = encodings["attention_mask"].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            predictions = torch.argmax(outputs.logits, dim=-1)
            
            # Extract predictions aligned with original tokens
            for j in range(len(batch_sents)):
                word_ids = encodings.word_ids(j)
                pred_tags = []
                previous_word_id = None
                for k, word_id in enumerate(word_ids):
                    if word_id is not None and word_id != previous_word_id:
                        pred_tags.append(id2label[predictions[j][k].item()])
                        previous_word_id = word_id
                all_preds.append(pred_tags)
                all_labels.append(batch_tags[j])
    
    precision = precision_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "report": classification_report(all_labels, all_preds)
    }

# Load test data
test_sents_raw, test_tags_raw = load_bio("test.txt")
test_sents_raw = preprocess(test_sents_raw)

# Evaluate BioBERT
print("📊 Evaluating BioBERT on test set...")
biobert_results = evaluate_biobert(biobert_model, tokenizer, test_sents_raw, test_tags_raw, id2label)

print("\n🔷 BioBERT Performance:\n")
print(f"Precision: {biobert_results['precision']:.4f}")
print(f"Recall:    {biobert_results['recall']:.4f}")
print(f"F1-score:  {biobert_results['f1']:.4f}")
print("\nClassification Report:")
print(biobert_results['report'])

# Compare with BiLSTM+CRF (existing model)
print("\n" + "="*60)
print("🔷 Comparison: BiLSTM+CRF vs BioBERT")
print("="*60)

# Evaluate existing BiLSTM+CRF on test set  
test_encodings = encode(test_sents_raw, test_tags_raw, word2idx, tag2idx, char2idx)
X_test, y_test, X_char_test = test_encodings[0], test_encodings[1], test_encodings[2]
X_test = pad(X_test, MAX_LEN)
y_test = pad(y_test, MAX_LEN)
X_char_test = pad_chars(X_char_test, MAX_LEN, MAX_WORD_LEN)
mask_test = create_mask(X_test)

test_loader = DataLoader(
    NERDataset(X_test, X_char_test, y_test, mask_test),
    batch_size=batch_size
)

bilstm_results = evaluate_model(model, test_loader, idx2tag, use_crf=True)

print(f"\n{'Model':<20} {'Precision':<12} {'Recall':<12} {'F1-score':<12}")
print("-" * 60)
print(f"{'BiLSTM+CRF':<20} {bilstm_results['precision']:<12.4f} {bilstm_results['recall']:<12.4f} {bilstm_results['f1']:<12.4f}")
print(f"{'BioBERT':<20} {biobert_results['precision']:<12.4f} {biobert_results['recall']:<12.4f} {biobert_results['f1']:<12.4f}")

# Determine best model
if biobert_results['f1'] > bilstm_results['f1']:
    print("\n🏆 BioBERT performs better!")
    best_model_name = "BioBERT"
else:
    print("\n🏆 BiLSTM+CRF performs better!")
    best_model_name = "BiLSTM+CRF"

print(f"\n✅ Best model for this dataset: {best_model_name}")


📊 Evaluating BioBERT on test set...


ValueError: Found input variables with inconsistent numbers of samples:
[327, 378, 406, 220, 348, 318, 283, 379, 1094, 254, 384, 588, 394, 422, 233, 381, 1053, 660, 795, 456]
[327, 368, 396, 219, 348, 318, 282, 349, 408, 254, 361, 376, 394, 364, 231, 381, 407, 369, 383, 391]

In [100]:
# -------------------------------
# Training + Validation
# -------------------------------
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

def run_epoch(loader, training=True):
    total_loss = 0

    if training:
        model.train()
    else:
        model.eval()

    for Xb, Xcb, yb, mb in loader:
        Xb, Xcb, yb, mb = (
            Xb.to(device),
            Xcb.to(device),
            yb.to(device),
            mb.to(device)
        )

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            loss = model(Xb, Xcb, tags=yb, mask=mb)

            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

# -------------------------------
# Main Training Loop
# -------------------------------
def train_model(epochs=15):
    best_val_loss = float("inf")

    for epoch in range(epochs):
        train_loss = run_epoch(train_loader, training=True)
        val_loss = run_epoch(val_loader, training=False)
        scheduler.step()

        print(f"Epoch {epoch+1}")
        print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "best_model.pt")
            print("âœ… Best model saved")

# -------------------------------
# Train
# -------------------------------
train_model(epochs=15)

# -------------------------------
# Load Best Model
# -------------------------------
model.load_state_dict(torch.load("best_model.pt"))
model.eval()

Epoch 1
Train Loss: 2035.8505 | Val Loss: 1390.9177
âœ… Best model saved
Epoch 2
Train Loss: 1300.0708 | Val Loss: 1117.2955
âœ… Best model saved


KeyboardInterrupt: 

Evaluate the Model

In [90]:
%pip install seqeval

import torch
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from collections import Counter

# -------------------------------
# Evaluate + Error Analysis
# -------------------------------
def evaluate_and_analyze(model, loader, idx2tag, sentences=None):
    model.eval()

    all_preds = []
    all_labels = []
    results = []   # (true_seq, pred_seq)

    with torch.no_grad():
        for Xb, Xcb, yb, mb in loader:
            Xb, Xcb, yb, mb = (
                Xb.to(device),
                Xcb.to(device),
                yb.to(device),
                mb.to(device)
            )

            preds = model(Xb, Xcb, mask=mb)

            for i in range(len(preds)):
                seq_len = mb[i].sum().item()

                pred_seq = preds[i][:seq_len]
                true_seq = yb[i][:seq_len].tolist()

                all_preds.append(pred_seq)
                all_labels.append(true_seq)
                results.append((true_seq, pred_seq))

    # -------------------------------
    # Convert indices â†’ tags
    # -------------------------------
    pred_tags = [[idx2tag[i] for i in seq] for seq in all_preds]
    true_tags = [[idx2tag[i] for i in seq] for seq in all_labels]

    # -------------------------------
    # 1. Evaluation Metrics
    # -------------------------------
    print("\nðŸ”· Classification Report:\n")
    print(classification_report(true_tags, pred_tags))

    precision = precision_score(true_tags, pred_tags)
    recall = recall_score(true_tags, pred_tags)
    f1 = f1_score(true_tags, pred_tags)

    print(f"\nPrecision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-score:  {f1:.4f}")

    # -------------------------------
    # Token-level Accuracy
    # -------------------------------
    correct = sum(p == t for seq_p, seq_t in zip(all_preds, all_labels) for p, t in zip(seq_p, seq_t))
    total = sum(len(seq) for seq in all_labels)
    acc = correct / total

    print(f"Token Accuracy: {acc:.4f}")

# Call the function
evaluate_and_analyze(model, val_loader, idx2tag)

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip



ðŸ”· Classification Report:



c:\Users\sanan\AppData\Local\Programs\Python\Python313\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: <pad> seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\sanan\AppData\Local\Programs\Python\Python313\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\sanan\AppData\Local\Programs\Python\Python313\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


                        precision    recall  f1-score   support

              Activity       0.00      0.00      0.00         4
        Administration       0.00      0.00      0.00        14
                   Age       0.00      0.00      0.00        20
                  Area       0.00      0.00      0.00         1
  Biological_attribute       0.00      0.00      0.00         5
  Biological_structure       0.01      0.01      0.01       316
        Clinical_event       0.00      0.00      0.00        71
                 Color       0.00      0.00      0.00         8
           Coreference       0.01      0.02      0.01        51
                  Date       0.00      0.00      0.00        48
  Detailed_description       0.00      0.00      0.00       333
  Diagnostic_procedure       0.00      0.00      0.00       431
      Disease_disorder       0.00      0.10      0.01        96
              Distance       0.00      0.00      0.00        12
                Dosage       0.00      

In [91]:
# Evaluate the model on validation data
evaluate_and_analyze(model, val_loader, idx2tag)


ðŸ”· Classification Report:

                        precision    recall  f1-score   support

              Activity       0.00      0.00      0.00         4
        Administration       0.00      0.00      0.00        14
                   Age       0.00      0.00      0.00        20
                  Area       0.00      0.00      0.00         1
  Biological_attribute       0.00      0.00      0.00         5
  Biological_structure       0.01      0.01      0.01       316
        Clinical_event       0.00      0.00      0.00        71
                 Color       0.00      0.00      0.00         8
           Coreference       0.01      0.02      0.01        51
                  Date       0.00      0.00      0.00        48
  Detailed_description       0.00      0.00      0.00       333
  Diagnostic_procedure       0.00      0.00      0.00       431
      Disease_disorder       0.00      0.10      0.01        96
              Distance       0.00      0.00      0.00        12
         

In [92]:
%pip install seqeval

import torch
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from collections import Counter

# -------------------------------
# Evaluate + Error Analysis
# -------------------------------
def evaluate_and_analyze(model, loader, idx2tag, sentences=None):
    model.eval()

    all_preds = []
    all_labels = []
    results = []   # (true_seq, pred_seq)

    with torch.no_grad():
        for Xb, Xcb, yb, mb in loader:
            Xb, Xcb, yb, mb = (
                Xb.to(device),
                Xcb.to(device),
                yb.to(device),
                mb.to(device)
            )

            preds = model(Xb, Xcb, mask=mb)

            for i in range(len(preds)):
                seq_len = mb[i].sum().item()

                pred_seq = preds[i][:seq_len]
                true_seq = yb[i][:seq_len].tolist()

                all_preds.append(pred_seq)
                all_labels.append(true_seq)
                results.append((true_seq, pred_seq))

    # -------------------------------
    # Convert indices â†’ tags
    # -------------------------------
    pred_tags = [[idx2tag[i] for i in seq] for seq in all_preds]
    true_tags = [[idx2tag[i] for i in seq] for seq in all_labels]

    # -------------------------------
    # 1. Evaluation Metrics
    # -------------------------------
    print("\nðŸ”· Classification Report:\n")
    print(classification_report(true_tags, pred_tags))

    precision = precision_score(true_tags, pred_tags)
    recall = recall_score(true_tags, pred_tags)
    f1 = f1_score(true_tags, pred_tags)

    print(f"\nPrecision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-score:  {f1:.4f}")

    # -------------------------------
    # Token-level Accuracy
    # -------------------------------
    correct = sum(p == t for seq_p, seq_t in zip(all_preds, all_labels) for p, t in zip(seq_p, seq_t))
    total = sum(len(seq) for seq in all_labels)
    acc = correct / total

    print(f"Token Accuracy: {acc:.4f}")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Error Analysis

In [93]:
def analyze_results(results, idx2tag, sentences=None):
    # -------------------------------
    # 2. Error Analysis

    # Token-level errors
    token_errors = [(idx2tag[t], idx2tag[p])
                    for (t_seq, p_seq) in results
                    for t, p in zip(t_seq, p_seq) if t != p]

    print(f"\nðŸ”· Total Token Errors: {len(token_errors)}")
    print("Sample Errors:")
    for e in token_errors[:10]:
        print(f"{e[0]} â†’ {e[1]}")

    # Confusion analysis
    confusion = Counter(token_errors)
    print("\nðŸ”· Top Confusions:")
    for (t, p), count in confusion.most_common(10):
        print(f"{t} â†’ {p}: {count}")

    # Entity-level errors
    fn = sum(1 for t_seq, p_seq in results for t, p in zip(t_seq, p_seq)
             if t.startswith("B") and p == "O")

    fp = sum(1 for t_seq, p_seq in results for t, p in zip(t_seq, p_seq)
             if t == "O" and p.startswith("B"))

    print(f"\nðŸ”· Entity-Level Errors:")
    print(f"Missed entities (FN): {fn}")
    print(f"Wrong entities (FP): {fp}")

    # Sentence-level errors (optional)
    if sentences:
        print("\nðŸ”· Sample Sentence Errors:")
        shown = 0
        for i, (true_seq, pred_seq) in enumerate(results):
            if true_seq != pred_seq:
                print("\n--- Sentence ---")
                for w, t, p in zip(sentences[i], true_seq, pred_seq):
                    print(f"{w:15} | {idx2tag[t]:10} | {idx2tag[p]}")
                shown += 1
                if shown >= 3:
                    break

    return None


Compare Models

In [94]:
import torch
import torch.nn as nn
import re
from torchcrf import CRF
from seqeval.metrics import f1_score, precision_score, recall_score

# -------------------------------
# Device
# -------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------------
# Shared Character Encoder
# -------------------------------
class CharEncoder(nn.Module):
    def __init__(self, char_vocab_size):
        super().__init__()
        self.embed = nn.Embedding(char_vocab_size, 30, padding_idx=0)
        self.lstm = nn.LSTM(30, 50, batch_first=True, bidirectional=True)

    def forward(self, X_char):
        B, L, W = X_char.size()
        X_char = X_char.view(-1, W)
        x = self.embed(X_char)
        _, (h, _) = self.lstm(x)
        x = torch.cat((h[0], h[1]), dim=1)
        return x.view(B, L, -1)

# -------------------------------
# Model 1: BiLSTM (Baseline)
# -------------------------------
class BiLSTM(nn.Module):
    def __init__(self, word_embedding, char_vocab_size, tagset_size):
        super().__init__()
        self.word_embed = word_embedding
        self.char_enc = CharEncoder(char_vocab_size)

        self.lstm = nn.LSTM(200, 100, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(200, tagset_size)

    def forward(self, X, X_char):
        w = self.word_embed(X)
        c = self.char_enc(X_char)
        x = torch.cat([w, c], dim=-1)
        x, _ = self.lstm(x)
        return self.fc(x)

# -------------------------------
# Model 2: BiLSTM + CRF
# -------------------------------
class BiLSTM_CRF(nn.Module):
    def __init__(self, word_embedding, char_vocab_size, tagset_size):
        super().__init__()
        self.word_embed = word_embedding
        self.char_enc = CharEncoder(char_vocab_size)

        self.lstm = nn.LSTM(200, 100, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(200, tagset_size)
        self.crf = CRF(tagset_size, batch_first=True)

    def forward(self, X, X_char, tags=None, mask=None):
        w = self.word_embed(X)
        c = self.char_enc(X_char)
        x = torch.cat([w, c], dim=-1)
        x, _ = self.lstm(x)
        emissions = self.fc(x)

        if tags is not None:
            return -self.crf(emissions, tags, mask=mask, reduction='mean')
        return self.crf.decode(emissions, mask=mask)

# -------------------------------
# Evaluation (shared)
# -------------------------------
def evaluate_model(model, loader, idx2tag, use_crf=True):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for Xb, Xcb, yb, mb in loader:
            Xb, Xcb, yb, mb = Xb.to(device), Xcb.to(device), yb.to(device), mb.to(device)

            if use_crf:
                preds = model(Xb, Xcb, mask=mb)
            else:
                logits = model(Xb, Xcb)
                preds = torch.argmax(logits, dim=-1)

            for i in range(len(preds)):
                seq_len = mb[i].sum().item()
                p = preds[i][:seq_len] if use_crf else preds[i][:seq_len].tolist()
                t = yb[i][:seq_len].tolist()

                all_preds.append(p)
                all_labels.append(t)

    pred_tags = [[idx2tag[i] for i in seq] for seq in all_preds]
    true_tags = [[idx2tag[i] for i in seq] for seq in all_labels]

    return {
        "precision": precision_score(true_tags, pred_tags),
        "recall": recall_score(true_tags, pred_tags),
        "f1": f1_score(true_tags, pred_tags)
    }

# -------------------------------
# Compare Models
# -------------------------------
def compare_models(results):
    print("\n🔷 Model Comparison\n")
    print(f"{'Model':15} {'Precision':10} {'Recall':10} {'F1-score':10}")
    for name, m in results.items():
        print(f"{name:15} {m['precision']:.4f}     {m['recall']:.4f}     {m['f1']:.4f}")

In [95]:
results = {}

# Create the BiLSTM baseline; this baseline must be trained separately before the comparison is meaningful.
bilstm_model = BiLSTM(
    word_embedding,
    char_vocab_size=len(char2idx),
    tagset_size=len(tag2idx)
).to(device)

# Use the trained CRF model already loaded in the notebook.
crf_model = model

results["BiLSTM"] = evaluate_model(bilstm_model, val_loader, idx2tag, use_crf=False)
results["BiLSTM+CRF"] = evaluate_model(crf_model, val_loader, idx2tag, use_crf=True)

compare_models(results)



🔷 Model Comparison

Model           Precision  Recall     F1-score  
BiLSTM          0.0021     0.0063     0.0032
BiLSTM+CRF      0.0018     0.0074     0.0029


Build Prediction System

In [96]:
# -------------------------------
# Prediction System
# -------------------------------
def tokenize(text):
    return re.findall(r'\w+|\S', text)

def clean_token(t):
    return re.sub(r'\d', '0', t.lower())

def encode_input(text, word2idx, char2idx, max_len, max_word_len):
    tokens = tokenize(text)
    clean = [clean_token(t) for t in tokens]

    word_ids = [word2idx.get(w, 1) for w in clean]
    char_ids = [[char2idx.get(c, 1) for c in w][:max_word_len] for w in clean]

    char_ids = [c + [0]*(max_word_len-len(c)) for c in char_ids]

    word_ids = word_ids[:max_len] + [0]*(max_len-len(word_ids))
    char_ids = char_ids[:max_len] + [[0]*max_word_len]*(max_len-len(char_ids))

    mask = [1 if w != 0 else 0 for w in word_ids]

    return tokens, torch.tensor([word_ids]), torch.tensor([char_ids]), torch.tensor([mask], dtype=torch.bool)

def extract_entities(tokens, tags):
    entities, ent, label = [], "", ""

    for tok, tag in zip(tokens, tags):
        if tag.startswith("B-"):
            if ent:
                entities.append((ent, label))
            ent, label = tok, tag[2:]
        elif tag.startswith("I-") and ent:
            ent += " " + tok
        else:
            if ent:
                entities.append((ent, label))
                ent, label = "", ""

    if ent:
        entities.append((ent, label))

    return entities

def predict_text(model, text, word2idx, char2idx, idx2tag, max_len, max_word_len):
    model.eval()

    tokens, X, Xc, mask = encode_input(text, word2idx, char2idx, max_len, max_word_len)
    X, Xc, mask = X.to(device), Xc.to(device), mask.to(device)

    with torch.no_grad():
        preds = model(X, Xc, mask=mask)

    tags = [idx2tag[i] for i in preds[0][:len(tokens)]]
    entities = extract_entities(tokens, tags)

    return tokens, tags, entities

In [97]:
sentence = "Patient has diabetes and takes metformin"

tokens, tags, entities = predict_text(
    crf_model,
    sentence,
    word2idx,
    char2idx,
    idx2tag,
    MAX_LEN,
    MAX_WORD_LEN
)

print("\nEntities:")
for e in entities:
    print(e)


Entities:
('Patient has', 'Disease_disorder')
('diabetes', 'Date')
('and takes metformin', 'Occupation')
